# Gate sensitive agent actions with an external checkpoint

Agents often need tools that can create side effects, such as exporting data, sending a message, or writing to an external system. For those tools, it can be useful to put a deterministic approval boundary immediately before the side effect.

This example shows one lightweight pattern:

1. Build a stable action object from the tool arguments.
2. Compute an action hash from canonical JSON.
3. Send the action and hash to an external checkpoint.
4. Allow, pause, or block before the side effect runs.

The checkpoint can be an internal policy service, a human review queue, or a governance provider such as OSuite.

In [ ]:
%pip install llama-index

## Create a stable action object

The action object should include the fields that matter for review and audit. Keep it stable: use explicit field names, avoid timestamps in the hash input, and serialize with sorted keys.

In [ ]:
import hashlib
import json
from typing import Any, Literal


CheckpointDecision = Literal["allow", "pause", "block"]


def canonical_action_hash(action: dict[str, Any]) -> str:
    """Return a stable hash for the action that reviewers approve."""
    encoded = json.dumps(
        action,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def build_export_action(
    *,
    customer_id: str,
    destination: str,
    requested_by: str,
) -> dict[str, Any]:
    return {
        "schema_version": "v1",
        "action_type": "customer_data_export",
        "resource": {"customer_id": customer_id},
        "destination": destination,
        "requested_by": requested_by,
    }

## Checkpoint before the side effect

`external_checkpoint` is intentionally a stub. In production, this is where you would call a policy engine, queue a human approval task, or record an audit decision.

In [ ]:
def external_checkpoint(
    *,
    action: dict[str, Any],
    action_hash: str,
) -> CheckpointDecision:
    """Replace this stub with your approval, policy, or review service."""
    if action["destination"].endswith("@example.com"):
        return "allow"
    if action["destination"].endswith("@review.example"):
        return "pause"
    return "block"


def perform_customer_export(
    *,
    customer_id: str,
    destination: str,
) -> str:
    """Placeholder for the real export/send/write operation."""
    return f"Exported customer {customer_id} to {destination}"

Now wrap the sensitive operation in a tool function. The tool constructs the exact action being requested, sends that stable object to the checkpoint, and only performs the side effect on `allow`.

In [ ]:
def guarded_customer_export(
    customer_id: str,
    destination: str,
    requested_by: str,
) -> str:
    """Export customer data after an external checkpoint approves it."""
    action = build_export_action(
        customer_id=customer_id,
        destination=destination,
        requested_by=requested_by,
    )
    action_hash = canonical_action_hash(action)
    decision = external_checkpoint(action=action, action_hash=action_hash)

    if decision == "allow":
        result = perform_customer_export(
            customer_id=customer_id,
            destination=destination,
        )
        return f"Allowed {action_hash}: {result}"

    if decision == "pause":
        return f"Paused {action_hash}: waiting for external approval"

    return f"Blocked {action_hash}: external checkpoint denied the action"

You can exercise each decision path without calling an LLM:

In [ ]:
print(
    guarded_customer_export(
        customer_id="cust_123",
        destination="ops@example.com",
        requested_by="agent_session_42",
    )
)
print(
    guarded_customer_export(
        customer_id="cust_123",
        destination="review@review.example",
        requested_by="agent_session_42",
    )
)
print(
    guarded_customer_export(
        customer_id="cust_123",
        destination="unknown@external.test",
        requested_by="agent_session_42",
    )
)

## Use it as a LlamaIndex agent tool

The same function can be exposed to a `FunctionAgent`. The important part is that the checkpoint lives inside the tool function, directly before the side effect, rather than relying only on prompting.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI


llm = OpenAI(model="gpt-4o-mini", api_key="sk-...")

agent = FunctionAgent(
    tools=[guarded_customer_export],
    llm=llm,
    system_prompt=(
        "You help operators export customer data. Use the export tool "
        "only when the user provides the customer, destination, and requester."
    ),
)

In [ ]:
response = await agent.run(
    user_msg=(
        "Export customer cust_123 to ops@example.com. "
        "The requester is agent_session_42."
    )
)
print(str(response))